# GISLR — Five-architecture benchmark: GRU, LSTM, BiLSTM, CNN, DNN

**What this notebook does.** Five architectures, trained and evaluated
identically, on the current canonical split — the SAME feature pipeline
(`base_v1`, ME-126/xy, 252-dim raw landmarks) and the SAME training regime
(copied verbatim from `gislr.training.json`'s `shared` block) for a direct
apples-to-apples comparison:

| arm | model class | recurrence | streaming |
|---|---|---|---|
| `gru` | `sb.recognize.architectures.StreamingGRU` | unidirectional | yes |
| `lstm` | `sb.recognize.architectures.StreamingLSTM` | unidirectional | yes |
| `bilstm` | `sb.recognize.architectures.BiLSTM` | bidirectional | **no** (offline-only) |
| `cnn` | `sb.recognize.architectures.CausalConv1D` | none (dilated causal conv) | yes |
| `dnn` | `sb.recognize.interp.models.LandmarkDNN` | **none** (per-frame MLP) | yes (memory-free, causal by construction) |

**Why `dnn` is new here.** `LandmarkDNN` has previously only been measured on
`landmark_interp_v1`'s full-543-landmark pipeline
(`gislr.1.models.landmark-importance.ipynb`) or the 922-dim curated pipeline
(`gislr.1.models.curated-features.ipynb`) — never on the same raw 252-dim
ME-126/xy pipeline the other four use here. Fed directly (`n_landmarks=126,
channels_per_landmark=2, angle_dim=0, relational_dim=0` — no angle/relational
blocks, since `base_v1` doesn't compute them), it becomes directly comparable
to `gru`/`lstm`/`bilstm`/`cnn` for the first time.

**`gru`/`lstm`/`bilstm`/`cnn` here are single-final-fit measurements**
(this track's convention — small internal-val carve-out for early stopping,
no k-fold, not written to `registry/`), run through this notebook so all
five architectures share identical code and eval, not because they lack a
canonical home: `gislr.1.models.training.ipynb` (TODO §4.3) is where the
*registry-tracked, resumable, canonically-evaluated* numbers for
`gru`/`lstm`/`bilstm`/`cnn1d` come from, across all three landmark subsets.
Use **that** notebook's numbers for the authoritative leaderboard; use
**this** notebook to see `dnn` sitting next to the other four under
identical conditions, and for the confidence-score metric below (§5),
which the canonical eval doesn't compute.

**New metric: mean true-class confidence.** Alongside top-1/3/5 (ranked
accuracy — was the correct class in the top N predictions?), this notebook
adds a rank-independent metric: **for every test sample, the probability the
model assigned to the *correct* class specifically** — e.g. true label
`sleep`, model outputs `dog: 0.9, cat: 0.65, sleep: 0.2, ...` → that sample's
score is **0.2**, regardless of `sleep`'s rank. Averaged per gloss (per
label) and overall. Unlike top-N, this is sensitive to *how* wrong or right
a prediction was, not just whether the correct class cleared a rank
threshold — a model that puts 0.49 on the true class every time it's
"wrong" scores very differently here than one that puts 0.01 on it, even
though both miss top-1 identically.

**`cnn`'s `num_layers` override to 5** — same fix as `gislr.training.json`'s
`cnn1d` (TODO §5.5.1): for `CausalConv1D`, `num_layers` counts dilated conv
*blocks*, not recurrent layers — the receptive field is `1 + (kernel-1) *
sum(2^i for i in range(num_layers))` = **125 frames at 5 blocks** (dilations
1,2,4,8,16, matching `MAX_SEQ_LEN=128`) but only **13 frames at the shared
`num_layers=2`**, far too short to see a whole sign.

**`bilstm` is OFFLINE-ONLY** — bidirectional means the backward pass reads
future frames, so it is never a deployment candidate (same constraint
`sb.recognize.architectures.BiLSTM`'s own docstring states). The other four
are streaming-viable.

**Pipeline stage vs. diagnostic.** Self-contained under
`sb.recognize.interp` (mirrors `gislr.1.models.bilstm-curated.ipynb`'s
pattern exactly), not part of `ARCHS`/the registry.

**Artifacts**

| Path | Content |
|---|---|
| `data/cache/gislr/features/base_v1/<key>/` | ME-126/xy raw feature cache (train + test), content-addressed, shared by all five arms |
| `data/cache/gislr/five_arch_benchmark_runs/<arm>/{final.pt,test_predictions.npz}` | per-arm artifacts (gitignored) |
| `data/cache/gislr/five_arch_benchmark_runs/topn_summary.csv` | top-1/3/5 + mean true-class confidence per arm |
| `data/cache/gislr/five_arch_benchmark_runs/per_gloss_confidence.csv` | per-gloss mean true-class confidence, all five arms |
| `docs/reports/five-arch-benchmark.md` | write-up, **after** this notebook runs |

In [ ]:
# ============================================================
# Imports, tunables, resolved paths
# ============================================================
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

from sb.core.paths import CACHE_DIR, EXPERIMENTS_DIR
from sb.core.subsets import SUBSETS
from sb.recognize.architectures import BiLSTM, CausalConv1D, StreamingGRU, StreamingLSTM
from sb.recognize.features import base_v1 as BFEAT
from sb.recognize.interp.models import LandmarkDNN
from sb.recognize.interp.train import (
    load_split_arrays,
    make_fold_loader,
    make_row_tracked_loader,
    predict_probs_indexed,
    train_fold,
)
from sb.recognize.sources import get_source

CONFIG_PATH = EXPERIMENTS_DIR / "recognition" / "configs" / "gislr.five-arch-benchmark.json"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

SEED = CONFIG["seed"]
HYP = CONFIG["shared"]  # every training hyperparameter comes from the config, never this cell
FINAL_VAL_FRACTION = CONFIG["final_fit_val_fraction"]
TOP_N = CONFIG["top_n"]
ARMS = ["gru", "lstm", "bilstm", "cnn", "dnn"]
ME_126 = SUBSETS["ME_126"]

RUN_ROOT = CACHE_DIR / "gislr" / "five_arch_benchmark_runs"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
for arm in ARMS:
    (RUN_ROOT / arm / "assets").mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"config: {CONFIG_PATH}")
print(f"device: {DEVICE}")
print(f"run artifacts: {RUN_ROOT}")
print(f"arms: {ARMS}")
print(f"feature pipeline: base_v1, ME-126/xy, 252-dim -- shared by all five arms")
print(f"epochs cap={HYP['epochs']} - batch_size={HYP['batch_size']} - top_n={TOP_N}")

## 1. Feature cache (one, shared by every arm)

Unlike `bilstm-curated.ipynb` (two different feature pipelines), every arm
here reads the exact same `base_v1` ME-126/xy cache — already built by
`gislr.1.models.training.ipynb`'s §4, reused unchanged.

In [ ]:
# ============================================================
# Canonical split + the one feature cache every arm shares
# ============================================================
ds = get_source("gislr")
DATA_DIR = ds.resolve_dir()
sign2idx = ds.label_map(DATA_DIR)
idx2sign = {v: k for k, v in sign2idx.items()}
N_CLASSES = len(sign2idx)

train_split, test_split = ds.canonical_split(DATA_DIR, sign2idx)
print(f"train.csv: {len(train_split)} videos - test.csv: {len(test_split)} videos - {N_CLASSES} classes")


def _tqdm_progress(label: str):
    state = {"bar": None, "last": 0}

    def _progress(done, total):
        if state["bar"] is None:
            state["bar"] = tqdm(total=total, desc=f"{label} feature cache", dynamic_ncols=True)
        state["bar"].update(done - state["last"])
        state["last"] = done
        if done >= total - 1:
            state["bar"].close()

    return _progress


tr_data_path, tr_off_path = BFEAT.build_cache(
    train_split, "train", ME_126, "xy", DATA_DIR, progress=_tqdm_progress("base_v1 train")
)
te_data_path, te_off_path = BFEAT.build_cache(
    test_split, "test", ME_126, "xy", DATA_DIR, progress=_tqdm_progress("base_v1 test")
)
FEATURE_DIM = len(ME_126) * 2  # xy only
print(f"base_v1 cache dir: {BFEAT.cache_dir(ME_126, 'xy', DATA_DIR)}  (feature_dim={FEATURE_DIM})")

## 2. Model factories

In [ ]:
# ============================================================
# Model factories -- every hyperparameter from CONFIG's shared block
# ============================================================
def hyp_for_arm(arm: str) -> dict:
    '''`shared` + this arm's `overrides` (if any) -- only `cnn` overrides
    anything (num_layers, see the title cell).'''
    return {**HYP, **CONFIG["architectures"][arm].get("overrides", {})}


def make_model(arm: str):
    arm_hyp = hyp_for_arm(arm)
    if arm == "gru":
        return StreamingGRU(input_size=FEATURE_DIM, hidden_size=arm_hyp["hidden_size"],
                            num_layers=arm_hyp["num_layers"], num_classes=N_CLASSES,
                            dropout=arm_hyp["dropout"])
    if arm == "lstm":
        return StreamingLSTM(input_size=FEATURE_DIM, hidden_size=arm_hyp["hidden_size"],
                             num_layers=arm_hyp["num_layers"], num_classes=N_CLASSES,
                             dropout=arm_hyp["dropout"])
    if arm == "bilstm":
        return BiLSTM(input_size=FEATURE_DIM, hidden_size=arm_hyp["hidden_size"],
                      num_layers=arm_hyp["num_layers"], num_classes=N_CLASSES,
                      dropout=arm_hyp["dropout"])
    if arm == "cnn":
        return CausalConv1D(input_size=FEATURE_DIM, hidden_size=arm_hyp["hidden_size"],
                            num_layers=arm_hyp["num_layers"], num_classes=N_CLASSES,
                            dropout=arm_hyp["dropout"])
    # dnn -- memory-free per-frame MLP, fed the same 252-dim raw vector directly
    # (no angle/relational blocks: base_v1 doesn't compute them)
    return LandmarkDNN(
        hidden_sizes=tuple(CONFIG["architectures"]["dnn"]["hidden_sizes"]),
        num_classes=N_CLASSES, dropout=arm_hyp["dropout"],
        n_landmarks=len(ME_126), channels_per_landmark=2, angle_dim=0, relational_dim=0,
    )


n_params = {arm: sum(p.numel() for p in make_model(arm).parameters()) for arm in ARMS}
pd.Series(n_params, name="n_params").to_frame()

## 3. Final fit (one model per arm, trained on (nearly) all of `train.csv`)

Single final fit, no k-fold — same convention `curated-features.ipynb`/
`bilstm-curated.ipynb` use. Internal-val carve-out drives early
stopping/LR only. `train_fold` dispatches on `arch == "dnn"` for the
per-frame loss/eval path (`sb.recognize.interp.train.run_epoch_dnn`);
every other arm here has the generic `forward(x, lengths) -> (B, C)`
contract and goes through `run_epoch_rnn` — that includes `cnn`, which
isn't recurrent at all but matches the same calling convention.

In [ ]:
# ============================================================
# Final fit: one model per arm
# ============================================================
tr_idx_final, val_idx_final = train_test_split(
    np.arange(len(train_split)), test_size=FINAL_VAL_FRACTION,
    stratify=train_split["label"], random_state=SEED,
)
print(f"final fit: {len(tr_idx_final)} train / {len(val_idx_final)} internal-val "
      f"(early-stopping signal only -- never a reported metric)")

train_data, train_off, train_labels = load_split_arrays(tr_data_path, tr_off_path, train_split)

final_models = {}
for arm in ARMS:
    arm_hyp = hyp_for_arm(arm)
    if arm_hyp != HYP:
        print(f"{arm}: overrides -> { {k: v for k, v in arm_hyp.items() if HYP.get(k) != v} }")

    ckpt = RUN_ROOT / arm / "final.pt"
    tr_loader = make_fold_loader(train_data, train_off, train_labels, tr_idx_final,
                                  arm_hyp["batch_size"], True, SEED, feature_dim=FEATURE_DIM)
    va_loader = make_fold_loader(train_data, train_off, train_labels, val_idx_final,
                                  arm_hyp["batch_size"], False, SEED, feature_dim=FEATURE_DIM)
    bar = tqdm(total=arm_hyp["epochs"], desc=f"{arm} - final fit", dynamic_ncols=True)
    model, history, best_val_acc, best_state = train_fold(
        arch=arm, model_fn=lambda a=arm: make_model(a), train_loader=tr_loader, val_loader=va_loader,
        hyp=arm_hyp, device=DEVICE, ckpt_path=ckpt, bar=bar, bar_prefix="final",
    )
    bar.close()
    model.load_state_dict(best_state)
    model.eval()
    final_models[arm] = model
    (RUN_ROOT / arm / "history.json").write_text(json.dumps(history))
    print(f"{arm}: final fit best internal-val acc {best_val_acc:.4f}")

## 4. Held-out test-set evaluation — top-N + mean true-class confidence

One pass per arm over the untouched canonical `test.csv`. Two metrics from
the same prediction matrix:

- **top-1/3/5** (ranked): was the correct class among the model's top N
  predictions?
- **mean true-class confidence** (rank-independent, new): the probability
  mass the model put on the *correct* class specifically, averaged over all
  test samples (**overall**) and separately per sign (**per-gloss**, §4b) —
  see the title cell for the worked example.

In [ ]:
# ============================================================
# ONE evaluation pass per final model, on the untouched canonical test.csv
# ============================================================
test_data, test_off, test_labels_arr = load_split_arrays(te_data_path, te_off_path, test_split)

arm_results = {}
topn_rows = []
for arm in ARMS:
    loader = make_row_tracked_loader(
        test_data, test_off, test_labels_arr, np.arange(len(test_split)),
        hyp_for_arm(arm)["batch_size"], feature_dim=FEATURE_DIM,
    )
    rows, probs, ret_labels = predict_probs_indexed(
        final_models[arm], arm, loader, DEVICE, desc=f"{arm} - test.csv eval"
    )
    order = np.argsort(rows)
    labels_ordered, probs_ordered = ret_labels[order], probs[order]

    ranked = np.argsort(-probs_ordered, axis=-1)
    for n in TOP_N:
        hit = (ranked[:, :n] == labels_ordered[:, None]).any(axis=1)
        topn_rows.append({"arm": arm, "top_n": n, "accuracy": float(hit.mean())})

    # mean true-class confidence: the probability mass on the CORRECT class,
    # regardless of its rank -- not a top-N accuracy at all, a separate axis
    true_class_conf = probs_ordered[np.arange(len(labels_ordered)), labels_ordered]

    arm_results[arm] = dict(labels=labels_ordered, probs=probs_ordered, ranked=ranked,
                             true_class_conf=true_class_conf)
    np.savez(RUN_ROOT / arm / "test_predictions.npz",
             labels=labels_ordered, probs=probs_ordered, ranked_top5=ranked[:, :5],
             true_class_conf=true_class_conf)

topn_df = pd.DataFrame(topn_rows).pivot(index="arm", columns="top_n", values="accuracy")
topn_df.columns = [f"top_{n}_acc" for n in topn_df.columns]
topn_df["mean_true_class_confidence"] = {
    arm: float(arm_results[arm]["true_class_conf"].mean()) for arm in ARMS
}
topn_df = topn_df.loc[ARMS]
topn_df.to_csv(RUN_ROOT / "topn_summary.csv")
topn_df

### 4b. Per-gloss mean true-class confidence

The same score (§4), broken out per sign rather than averaged over the
whole test set — which glosses does each architecture actually "believe
in" vs merely rank correctly often enough?

In [ ]:
# ============================================================
# Per-gloss mean true-class confidence, every arm
# ============================================================
per_gloss_frames = []
for arm in ARMS:
    labels_ordered = arm_results[arm]["labels"]
    true_class_conf = arm_results[arm]["true_class_conf"]
    df = pd.DataFrame({"label": labels_ordered, "confidence": true_class_conf})
    g = df.groupby("label")["confidence"].agg(["mean", "std", "count"]).reset_index()
    g["sign"] = g["label"].map(idx2sign)
    g["arm"] = arm
    per_gloss_frames.append(g[["arm", "sign", "mean", "std", "count"]])

per_gloss_confidence = pd.concat(per_gloss_frames, ignore_index=True)
per_gloss_confidence.to_csv(RUN_ROOT / "per_gloss_confidence.csv", index=False)

print("5 lowest-confidence glosses per arm (the model 'believes in' these least, "
      "even when it still ranks them correctly at top-1):")
for arm in ARMS:
    sub = per_gloss_confidence[per_gloss_confidence["arm"] == arm].sort_values("mean")
    print(f"\n=== {arm} ===")
    print(sub.head(5)[["sign", "mean", "std", "count"]].to_string(index=False))

## 5. Confusable-pair check

Same 16 documented pairs as `curated-features.ipynb` §5 /
`docs/reports/pair-similarity.md`.

In [ ]:
# ============================================================
# Confusable-pair rank-recovery, per arm
# ============================================================
CONFUSABLE_PAIRS = [
    ("awake", "wake"), ("lips", "mouth"), ("hear", "listen"), ("pen", "pencil"),
    ("gift", "give"), ("cut", "scissors"), ("finger", "wait"), ("stay", "that"),
    ("duck", "goose"), ("animal", "have"), ("cat", "kitty"), ("sleep", "sleepy"),
    ("nap", "sleep"), ("bed", "bedroom"), ("dry", "dryer"), ("look", "see"),
]

for arm in ARMS:
    labels_ordered = arm_results[arm]["labels"]
    ranked = arm_results[arm]["ranked"]
    print(f"\n=== {arm} ===")
    rows = []
    for a, b in CONFUSABLE_PAIRS:
        if a not in sign2idx or b not in sign2idx:
            continue
        for true_sign, other_sign in ((a, b), (b, a)):
            true_idx, other_idx = sign2idx[true_sign], sign2idx[other_sign]
            mask = labels_ordered == true_idx
            if mask.sum() == 0:
                continue
            r = ranked[mask]
            top1 = (r[:, 0] == true_idx).mean()
            top1_is_other = (r[:, 0] == other_idx).mean()
            rows.append({"true_sign": true_sign, "confused_with": other_sign,
                        "n": int(mask.sum()), "top1_acc": top1,
                        "rank1_is_other": top1_is_other})
    print(pd.DataFrame(rows).to_string(index=False))

## 6. Comparison against the registry leaderboard

Every canonical registry run on the current split (`registry/index.csv`,
queried live so this table can't go stale).

In [ ]:
# ============================================================
# Situate this notebook's numbers against the rest of the current-split leaderboard
# ============================================================
from sb.mlops.query import query_runs

canonical_current_split = query_runs(
    where="dataset = 'gislr' AND split.n_val = 18896", order_by="accuracy DESC"
)
print("Canonical registry runs on the current split:")
print(canonical_current_split[["architecture", "subset", "coords", "accuracy"]].to_string(index=False)
      if len(canonical_current_split) else "(none)")

print()
print("This notebook, all five arms:")
print(topn_df)

## 7. Verdict

Run 2026-09-21 (all five arms, 0 failures). Full write-up: `docs/reports/five-arch-benchmark.md`.

**Held-out `test.csv` top-1**: `bilstm` 0.7392 (offline-only) > `gru` 0.7380 > `lstm` 0.7286 >
`cnn` 0.6696 > `dnn` 0.6485. Among streaming-viable arms, `gru` leads. Every arm sits at or
below the canonical `gru`/ME_132/xy registry leaderboard entry (0.7517) — no architecture here
beats the existing streaming baseline.

**Mean true-class confidence is the standout result**: `dnn` scores 0.2387, a quarter of every
other arm (0.59–0.61), despite top-3/5 accuracy in the same range as `cnn` — ranked accuracy and
confidence decouple for the per-frame, softmax-averaged architecture. `give`/`gift` is the
hardest confusable pair and `give` the lowest-confidence gloss for every single arm — the same
sign flagged as hardest by two independent metrics.

`cnn` and `dnn` barely overfit (train/val gap 0.036 and 0.005 vs 0.17–0.25 for the recurrent
arms) but underfit instead — running far more epochs without ever closing on the recurrent arms’
accuracy. See the report for the full breakdown, the registry cross-check, and `dnn`’s
across-pipeline comparison.


In [ ]:
# ============================================================
# Verdict table (fill in narrative above after running)
# ============================================================
print(topn_df)
print()
print("Params:", n_params)
print()
print("Artifacts:")
for arm in ARMS:
    print(f"  {RUN_ROOT / arm / 'final.pt'}")
    print(f"  {RUN_ROOT / arm / 'test_predictions.npz'}")
print(f"  {RUN_ROOT / 'topn_summary.csv'}")
print(f"  {RUN_ROOT / 'per_gloss_confidence.csv'}")